# Market Regime Detection System — Data Exploration

## Section 1: Imports and Setup

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path for imports
sys.path.insert(0, "..")

# Configure matplotlib
plt.style.use("seaborn-v0_8-whitegrid")
plt.rcParams["figure.figsize"] = (14, 5)

# Load features
features_path = Path("../data/processed/features.parquet")
features_df = pd.read_parquet(features_path)

print(f"Features loaded: {features_df.shape}")
print(f"Date range: {features_df.index.min()} to {features_df.index.max()}")
print(f"Columns: {list(features_df.columns[:5])} ... (showing first 5 of {len(features_df.columns)})")
print("\nSection 1 complete.")

ImportError: DLL load failed while importing _imaging: An Application Control policy has blocked this file.

## Section 2: Raw Feature Distributions

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

features_to_plot = [
    "SPY_return_1d",
    "SPY_vol_20d",
    "SPY_rsi_14",
    "vix_level",
    "bond_equity_ratio",
    "SPY_macd",
]

for idx, feature in enumerate(features_to_plot):
    axes[idx].hist(features_df[feature].dropna(), bins=50, edgecolor="black", alpha=0.7)
    axes[idx].set_title(feature, fontsize=12, fontweight="bold")
    axes[idx].set_xlabel("Value")
    axes[idx].set_ylabel("Frequency")

plt.tight_layout()
plt.show()
print("Section 2 complete: Raw feature distributions plotted.")

## Section 3: SPY Price Proxy Over Time

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

# Plot returns
ax.plot(features_df.index, features_df["SPY_return_1d"], label="SPY 1-day return", linewidth=1, color="blue")
ax.axhline(y=0, color="black", linestyle="--", linewidth=1)

# Shade panic periods (return < -0.03)
panic_mask = features_df["SPY_return_1d"] < -0.03
panic_periods = features_df[panic_mask]
for idx in panic_periods.index:
    ax.axvspan(idx, idx + pd.Timedelta(days=1), alpha=0.2, color="red")

ax.set_title("SPY daily returns — panic periods highlighted", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Daily Return")
ax.legend()
plt.tight_layout()
plt.show()
print("Section 3 complete: SPY returns with panic shading.")

## Section 4: VIX Over Time

In [ ]:
# Load raw VIX data
vix_path = Path("../data/raw/VIX.parquet")
vix_df = pd.read_parquet(vix_path)

fig, ax = plt.subplots(figsize=(14, 5))

# Plot VIX
ax.plot(vix_df.index, vix_df["Close"], label="VIX Close", linewidth=1.5, color="darkblue")

# Add regime threshold lines
ax.axhline(y=15, color="green", linestyle="--", linewidth=2, label="Growth threshold")
ax.axhline(y=30, color="red", linestyle="--", linewidth=2, label="Panic threshold")

# Shade panic regions (VIX > 30)
panic_vix_mask = vix_df["Close"] > 30
panic_vix_periods = vix_df[panic_vix_mask]
for idx in panic_vix_periods.index:
    ax.axvspan(idx, idx + pd.Timedelta(days=1), alpha=0.15, color="red")

ax.set_title("VIX level over time with regime thresholds", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("VIX Level")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()
print("Section 4 complete: VIX with regime thresholds.")

## Section 5: Normalised Features Check

In [ ]:
# Load rolling statistics
rolling_mean_path = Path("../data/processed/rolling_mean.parquet")
rolling_std_path = Path("../data/processed/rolling_std.parquet")

rolling_mean = pd.read_parquet(rolling_mean_path)
rolling_std = pd.read_parquet(rolling_std_path)

# Compute z-scored versions
spy_return_norm = (features_df["SPY_return_1d"] - rolling_mean["SPY_return_1d"]) / rolling_std["SPY_return_1d"]
vix_level_norm = (features_df["vix_level"] - rolling_mean["vix_level"]) / rolling_std["vix_level"]
bond_equity_norm = (features_df["bond_equity_ratio"] - rolling_mean["bond_equity_ratio"]) / rolling_std["bond_equity_ratio"]

fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(features_df.index, spy_return_norm, label="SPY return (z-scored)", linewidth=1, alpha=0.8)
ax.plot(features_df.index, vix_level_norm, label="VIX level (z-scored)", linewidth=1, alpha=0.8)
ax.plot(features_df.index, bond_equity_norm, label="Bond/equity ratio (z-scored)", linewidth=1, alpha=0.8)

ax.axhline(y=0, color="black", linestyle="--", linewidth=1)
ax.set_title("Normalised features — should hover around zero", fontsize=14, fontweight="bold")
ax.set_xlabel("Date")
ax.set_ylabel("Z-Score")
ax.legend()
plt.tight_layout()
plt.show()
print("Section 5 complete: Normalised features verified.")

## Section 6: Correlation Heatmap

In [ ]:
corr_matrix = features_df.corr()

fig, ax = plt.subplots(figsize=(16, 12))
sns.heatmap(
    corr_matrix,
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    square=True,
    ax=ax,
    cbar_kws={"label": "Correlation"},
)
ax.set_title("Feature correlation matrix", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()
print("Section 6 complete: Correlation heatmap generated.")

## Section 7: Train/Val/Test Split Visualisation

In [ ]:
# Load split data
x_train = np.load(Path("../data/processed/X_train.npy"), allow_pickle=False)
x_val = np.load(Path("../data/processed/X_val.npy"), allow_pickle=False)
x_test = np.load(Path("../data/processed/X_test.npy"), allow_pickle=False)

dates_train = np.load(Path("../data/processed/dates_train.npy"), allow_pickle=False)
dates_val = np.load(Path("../data/processed/dates_val.npy"), allow_pickle=False)
dates_test = np.load(Path("../data/processed/dates_test.npy"), allow_pickle=False)

# Convert to DatetimeIndex
train_dates = pd.to_datetime(dates_train)
val_dates = pd.to_datetime(dates_val)
test_dates = pd.to_datetime(dates_test)

fig, ax = plt.subplots(figsize=(14, 6))

# Plot horizontal bars for each split
ax.barh(2, (train_dates.max() - train_dates.min()).days, left=train_dates.min(), height=0.5, color="blue", label="Train", alpha=0.7)
ax.barh(1, (val_dates.max() - val_dates.min()).days, left=val_dates.min(), height=0.5, color="orange", label="Validation", alpha=0.7)
ax.barh(0, (test_dates.max() - test_dates.min()).days, left=test_dates.min(), height=0.5, color="green", label="Test", alpha=0.7)

# Add text labels
ax.text(train_dates.min(), 2.35, f"Train: {train_dates.min().date()} to {train_dates.max().date()} ({len(train_dates)} samples)", fontsize=9, va="center")
ax.text(val_dates.min(), 1.35, f"Val: {val_dates.min().date()} to {val_dates.max().date()} ({len(val_dates)} samples)", fontsize=9, va="center")
ax.text(test_dates.min(), 0.35, f"Test: {test_dates.min().date()} to {test_dates.max().date()} ({len(test_dates)} samples)", fontsize=9, va="center")

ax.set_yticks([0, 1, 2])
ax.set_yticklabels(["Test", "Validation", "Train"])
ax.set_xlabel("Date")
ax.set_title("Train / Val / Test split by date", fontsize=14, fontweight="bold")
ax.legend(loc="upper right")
plt.tight_layout()
plt.show()
print("Section 7 complete: Train/Val/Test split visualised.")

## Section 8: Window Shape Confirmation

In [ ]:
print(f"X_train shape: {x_train.shape}")
print(f"X_val shape: {x_val.shape}")
print(f"X_test shape: {x_test.shape}")
print("\nEach sample is a sequence of 60 days × 50 features")

# Plot first window from training set
first_window = x_train[0]  # Shape: (60, 50)

fig, ax = plt.subplots(figsize=(14, 8))
im = ax.imshow(first_window, cmap="viridis", aspect="auto")
ax.set_xlabel("Features", fontsize=12)
ax.set_ylabel("Days in window", fontsize=12)
ax.set_title("Example input window — one LSTM sample", fontsize=14, fontweight="bold")
cbar = plt.colorbar(im, ax=ax)
cbar.set_label("Normalised Value", fontsize=10)
plt.tight_layout()
plt.show()
print("\nSection 8 complete: Window shape and example visualised.")